In [8]:
# Notion page: https://www.notion.so/lavrovs/St-Simple-task-Linearisator-ada2eed5cdbb422886222b77d225c516



In [9]:
from IPython.display import Markdown, display
# def make_pre(text):
#     return f"<pre>{text}</pre>"
def printmd(string):
#     display(Markdown(make_pre(string)))
    display(Markdown(string))
    
# printmd("**bold text**")
# printmd("~bold text~")

def make_bold(text):
    return f"**{text}**"
def make_strikethrough(text):
    return f"~{text}~"

In [22]:
import logging as logger
# initialisation
class TaskTreeNode:
    def __init__(self, name, content="", completed=False, children=None, parent=None, selected=False, page_data=None):
        self.name = name
        self.content = content
        self.completed = completed
        if children is None:
            self.children = []
        else:
            self.children = children 
        self.parent = parent
        self.selected = selected
        self.page_data = page_data
        
    def size(self):
        return 1 + sum([c.size() for c in self.children])
        
    def __repr__(self, tab="  ", depth=0, expand_selected_only=False, expand_depth=None):
        res = tab*depth
        res += '[x]' if self.completed else ' * ' if self.selected else ' - '
        res += make_strikethrough(self.name) if self.completed else make_bold(self.name) if self.selected else self.name
        if self.selected:
            if self.content:
                # display content
                res += f' ({self.content})' # .replace('\n')
            if self.page_data is not None:
                res += f" https://www.notion.so/{self.page_data.id.replace('-','')}"
        # children
        if expand_selected_only and not self.selected:
            if expand_depth is None or depth >= expand_depth:
                size = self.size()
                if size > 5:
                    res += f' (**{size}**)'
                elif size > 1: 
                    res += f' ({size})'
                res += '  \\\n'
                return res
        res += '  \n'
        for c in self.children:
            res += c.__repr__(tab=tab, depth=depth+1, expand_selected_only=expand_selected_only, expand_depth=expand_depth)
        return res



class TaskTree:
    """
    Displaying:
    tree.show_full()
    tree.show_selected()
    tree.focus_selected()
    
    Navigation:
    tree.select(name) - select task by name or block_id
    tree.select_next(depth) - Jump to a next task, specified levels up from current selection
    tree.select_first(depth) - Jump to a first incomplete task, specified levels deep in current selection
    tree.select_random()
    
    Editing the tree:
    tree.load_page(url)
    """
    def __init__(self, nodes, name='TaskTree ', auto_show_selected=True):
        if not isinstance(nodes, list):
            nodes = [nodes]
        self._root = TaskTreeNode(name=name, children=nodes)
        
        if len(nodes) == 1:
            self._selected_node = nodes[0]
        else: 
            self._selected_node = None
        
        self._node_cache = dict()
        self._build_node_cache()
        
        self._tab = '\t'
        self.auto_show_selected = auto_show_selected
    
    def _build_node_cache(self):
        def dfs(node):
            self._node_cache[node.name] = node
            if node.page_data:
                self._node_cache[node.page_data.id] = node
            for c in node.children:
                dfs(c)
        dfs(self.root)
        
#     def linearise_selected():
#         # todo: show deepest first INCOMPLETE task from each child
#         res = []
#         selected = self._selected_node
#         for child in selected.children:
#             temp = child
#             path = child.name
#             incomplete_children = [c for c in temp.children if not c.complete]
#             while incomplete_children:
#                 temp = incomplete_children[0]
#                 path += f'/{temp.name}''
    
    @property
    def root(self):
        if len(self._root.children) == 1:
            return self._root.children[0]
        else:
            return self._root
    
    # Displaying
    
    def __repr__(self):
        return self.root.__repr__(tab=self._tab)
    
    def show_full(self):
        printmd(self.root.__repr__(tab='&emsp;&emsp;'))
    
    def focus_selected(self, full_depth = 1): # alternative tasks for current selection
        """
        full_depth: for this amount of levels starting from the root - show all possible candidates for the level.
        After that show only the first one
        """
        
        text = self._selected_node.__repr__(tab="&emsp;&emsp;", expand_selected_only=True, expand_depth=full_depth)
        printmd(text)
    
    def show_selected(self, full_depth = 1): # full page tree
        """
        full_depth: for this amount of levels starting from the root - show all possible candidates for the level.
        After that show only the first one
        """
        text = self.root.__repr__(tab="&emsp;&emsp;", expand_selected_only=True, expand_depth=full_depth)
        printmd(text)

    ### Selection, navigation
    
    @staticmethod
    def _mark_node_selection_status(node, selected):
        while node is not None:
            node.selected = selected
            node = node.parent
    
    def _select_node(self, node):
        # deselect
        self._mark_node_selection_status(self._selected_node, False)
        self._selected_node = node
        # select
        self._mark_node_selection_status(self._selected_node, True)
        
    def select(self, node_name):
        if node_name not in self._node_cache:
            raise KeyError(f"node {node_name} not found in TaskTree")
        self._select_node(self._node_cache[node_name])
        if self.auto_show_selected:
            self.show_selected()
    
    def select_next(self, depth=0):
        node = self._selected_node
        parent = node.parent
        
        for i in range(depth):
            if node.parent is not None:
                node = parent
                parent = node.parent
            else:
                raise Exception('Depth exceeded: {i}')
        
        # find next incomplete task
        index = parent.children.index(node)
        index += 1
        while index >= len(parent.children) or parent.children[index].completed:
            if index >= len(parent.children):
                node = parent
                parent = node.parent
                
                index = parent.children.index(node)
            index += 1
            
        # select
        self._select_node(parent.children[index])
        if self.auto_show_selected:
            self.show_selected()
        
    def select_first(self, depth=1): # selects first incomplete subtask of current selection
        selected = self._selected_node
        
        index = 0
        print(index >= len(selected.children))
        while index >= len(selected.children) or selected.children[index].completed:
            if index >= len(selected.children):
                raise Exception("No children available to go deeper")
            else:
                index += 1
        self._select_node(selected.children[index])
        
        if depth > 1:
            self.select_first(depth = depth - 1)
        else:
            if self.auto_show_selected:
                self.show_selected()
    
    def select_random(self):
        import random
        self._select_node(random.choice([n for n in self._node_cache.values() if not n.completed]))
        if self.auto_show_selected:
            self.show_selected()
    
    # Help
    
    def help(self):
        print(self.__doc__)
        
    # modifying the tree:
    def load_page(page_id=None, url=None):
        if page_id.startswith('http'):
            url = page_id
        if url is not None:
            page_id = parse_url(url)
        self._root.children.append(load_the_tree(page_id))
        logger.info("Page load successful")
        
new_node = {
    'name': 'Simple task linearisator',
    'children':[
       {'name':'Data model', 'completed':True},
       {'name':'Hihlight selected on repr'},
    ]
}

    
sample_data = {"name": "17 Sep 2022",
               "content": "Tasks I want to do today",
               "completed": False,
               "children": [
                   {'name': 'Simple task linearisator',
                   'children':[
                       {'name':'Data model', 'completed':True},
                       {'name':'Hihlight selected on repr'},
                       {'name':'Display repr as md'},
                       {'name': 'Notion',
                        'children':[
                           {'name':'Create Notion Template', 'content':'help me do \n and test the check'},
                           {'name':'Parse from Notion'},
                        ]},
                       {'name': 'Navigation',
                        'children':[
                            {'name': 'Show candidates'},
                           {'name':'Select next'},
                           {'name':'Empty?'},
                        ]},
                       {'name':'Data 2'},
                       {'name':'asdasd'},
                       {'name':'Parse from Notion 3'},
                   ]
                   },                   
                   {'name': 'Yoga intensive',
                   'children':[
                       {'name':'dm', 'completed':True},
                       {'name':'hs'},
                       {'name':'dr'},
                       {'name': 'n',
                        'children':[
                           {'name':'cnt', 'content':'help me do \n and test the check'},
                           {'name':'pfn'},
                        ]},
                       {'name': 'nav',
                        'children':[
                            {'name': 'sc'},
                           {'name':'st'},
                           {'name':'e'},
                        ]},
                       {'name':'Data 3'},
                       {'name':'asdasd 2'},
                       {'name':'Parse from Notion'},
                   ]},
                   {'name': 'Weekly review'},
                   {'name': 'Jarvis new features'},
                   {'name': 'Particles visualisation'}
               ]
              }

def parse_data(source_data):
    if 'children' not in source_data:
        return TaskTreeNode(**source_data)
    else:
        children = [parse_data(c) for c in source_data['children']]
        result = TaskTreeNode(**{k:v for k,v in source_data.items() if k != 'children'}, children=children)
        for child in children:
            child.parent = result
        return result

sample_task_tree = TaskTree(parse_data(sample_data))



In [23]:
# printmd(repr(sample_task_tree))
sample_task_tree.select('Create Notion Template')
# sample_task_tree

 * **17 Sep 2022** (Tasks I want to do today)  
&emsp;&emsp; * **Simple task linearisator**  
&emsp;&emsp;&emsp;&emsp;[x]~Data model~  \
&emsp;&emsp;&emsp;&emsp; - Hihlight selected on repr  \
&emsp;&emsp;&emsp;&emsp; - Display repr as md  \
&emsp;&emsp;&emsp;&emsp; * **Notion**  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp; * **Create Notion Template** (help me do 
 and test the check)  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp; - Parse from Notion  \
&emsp;&emsp;&emsp;&emsp; - Navigation (4)  \
&emsp;&emsp;&emsp;&emsp; - Data 2  \
&emsp;&emsp;&emsp;&emsp; - asdasd  \
&emsp;&emsp;&emsp;&emsp; - Parse from Notion 3  \
&emsp;&emsp; - Yoga intensive (**14**)  \
&emsp;&emsp; - Weekly review  \
&emsp;&emsp; - Jarvis new features  \
&emsp;&emsp; - Particles visualisation  \


In [24]:
# sample_task_tree.select_next()
# sample_task_tree.select_first()
sample_task_tree.focus_selected()

 * **Create Notion Template** (help me do 
 and test the check)  


In [25]:
from defaultenv import env
notion_api_key = env('JARVIS_NOTION_TOKEN')
page_to_parse = 'https://www.notion.so/lavrovs/STL_17-sep-22-483b8263ec2b48fa9bb941ba6e497516'

from pprint import pprint

from notion_assistant.jarvis.enhanced_notion_client import EnhancedNotionClient, get_notion_client
def parse_url(url):
    return url.rsplit('-', 1)[-1]

def load_notion_page(url, token):
    page_id = parse_url(url)
    client = get_notion_client(token)
    return client.get_page(page_id)


nc = get_notion_client(token=notion_api_key)

def get_metadata(page_id):
    return DotDict(nc.client.pages.retrieve(page_id))

def get_content(page_id):
    return DotDict(nc.client.blocks.retrieve(page_id))

def get_children(page_id):
    return DotDict(nc.client.blocks.children.list(page_id))

In [26]:
class DotList(list):
    def __getitem__(self, key):
        val = list.__getitem__(self, key)
        if isinstance(val, dict):
            return DotDict(val)
        if isinstance(val, list):
            return DotList(val)
        return val

    def __iter__(self):
        self.__iter_pos = -1
        return self

    def __next__(self):
        self.__iter_pos += 1
        if self.__iter_pos < len(self):
            return self[self.__iter_pos]
        else:
            raise StopIteration
    
class DotDict(dict):
    def __getattr__(self, key):
        val = self[key]
        if isinstance(val, dict):
            return DotDict(val)
        if isinstance(val, list):
            return DotList(val)
        return val
    
    def __getitem__(self, key):
        val = dict.__getitem__(self, key)
        if isinstance(val, dict):
            return DotDict(val)
        if isinstance(val, list):
            return DotList(val)
        return val

# page_data = get_metadata(root_page_id)


In [27]:
def is_mention(block_data):
    return any(['mention' in ann for ann in block_data.rich_text])

def get_mention_page_id(block_data):
    mention_page_id = None
    for ann in block_data.rich_text:
        if 'mention' in ann:
            if mention_page_id is not None:
                #warning
                logger.warning(f"Multiple page mentions in one block! {mention_page_id} and {ann.mention.page.id}")
            mention_page_id = ann.mention.page.id
    return mention_page_id

def page_is_constructor(block_data):
    constructor_pages = {'3a9625034ef746189508bb180fff2e53'}
    # hardcode
    for ann in block_data:
        if 'mention' in ann:
            if ann['mention'] in constructor_pages:
                return True
            else:
                page_id = ann.mention.page.id
                page_data = get_metadata(page_id)
                name = ''.join([c.plain_text for c in page_data.properties.title.title]).strip(),
                # load mention data
                if 'constructor' in name or 'Constructor' in name:
                    logger.warning(f"New constructor page found: {page_id}")
                    return True
    return False
                    
        

def load_the_tree(page_id, is_block = False):
    if is_block:
        page_data = get_content(page_id)
        block_id = page_data.id
        block_type = page_data.type
        block_data = page_data[block_type]
        text_parts = block_data.rich_text
        node = TaskTreeNode(
            name = ''.join([t.plain_text for t in text_parts]).strip(),
            page_data = page_data
        )
    else:
        page_data = get_metadata(page_id)
        node = TaskTreeNode(
            name = ''.join([c.plain_text for c in page_data.properties.title.title]).strip(),
            page_data = page_data
        )
    
    # load children
    children_data = get_children(page_id)
    
    content_parts = []
    children = []
    for child_data in children_data.results:
        block_id = child_data.id
        block_type = child_data.type
        block_data = child_data[block_type]
        # filter out 'STL_constructor block'
        if page_is_constructor(block_data):
            continue
        
        # todo: support other block types: toggle and mention (simple block). Optionally - page.
        if block_type == 'to_do': # todo: add toggle, list and simple block support
            # todo: if a block has children - ignore the mention
            # todo: save the mention for future for optional loading
            
            if is_mention(block_data):
                # if it's a mention - process page
                mention_page_id = get_mention_page_id(block_data)
                child = load_the_tree(mention_page_id, is_block=False)
            else:
                # if it's a simple block
                child = load_the_tree(block_id, is_block=True)
                
            child.completed = block_data.checked 
            child.parent = node
            children.append(child)
        else:
            # add to content
            text_parts = block_data.rich_text
            text = ''.join([t.plain_text for t in text_parts])
            if text.strip():
                content_parts.append(text.strip())
    
    content = '\n'.join(content_parts)
    node.content = content
    
    node.children = children
    return node
    
    
    
    


In [ ]:
root_page_id = parse_url(page_to_parse)    
root = load_the_tree(root_page_id)
# ~

In [28]:
tree = TaskTree(nodes=root)
tree.select('parse simple checkboxes')
# tree.show_full()

 * **STL_17 sep 22** (Untitled) https://www.notion.so/483b8263ec2b48fa9bb941ba6e497516  
&emsp;&emsp;[x]~Duolingo~  \
&emsp;&emsp; - T.assistant  \
&emsp;&emsp; - Yoga Intensive  \
&emsp;&emsp; - Jarvis  \
&emsp;&emsp; * **Simple task lineariser** (Legend. Idea
Notebook: http://localhost:8888/notebooks/notion_assistant/dev/notebooks/Simple task linearisator-Copy1.ipynb
Key features
Find better usages
Improve functionality
Bugfixes
How to use this tool?
Untitled) https://www.notion.so/3299efe7e2264018a64e050c3e50f82c  
&emsp;&emsp;&emsp;&emsp;[x]~stl Notion~ (Untitled) https://www.notion.so/4baf8763a553479fb9211be85fbe4ed3  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~Create template~  \
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~Load page from notion~ https://www.notion.so/bcc06d39810f4550b3731ec57a2a404c  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~parse page from notion~ https://www.notion.so/fda4740669c242fea052523cf98d6d6b  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~cleanup redundant sections (stl constructor)~  \
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~parse simple checkboxes~ https://www.notion.so/a2e0f2e195074703b32ea89cb83db0fb  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~parse checkbox status → completed~  \
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~parse sub-pages~  \
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~parse recursive children checkboxes~  \
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~store content~  \
&emsp;&emsp;&emsp;&emsp;[x]~stl navigation~ (5)  \
&emsp;&emsp; - Weekly review  \
&emsp;&emsp; - Particles  \


In [30]:
tree.select_random()

 * **STL_17 sep 22** (Untitled) https://www.notion.so/483b8263ec2b48fa9bb941ba6e497516  
&emsp;&emsp;[x]~Duolingo~  \
&emsp;&emsp; - T.assistant  \
&emsp;&emsp; - Yoga Intensive  \
&emsp;&emsp; - Jarvis  \
&emsp;&emsp; * **Simple task lineariser** (Legend. Idea
Notebook: http://localhost:8888/notebooks/notion_assistant/dev/notebooks/Simple task linearisator-Copy1.ipynb
Key features
Find better usages
Improve functionality
Bugfixes
How to use this tool?
Untitled) https://www.notion.so/3299efe7e2264018a64e050c3e50f82c  
&emsp;&emsp;&emsp;&emsp;[x]~stl Notion~ (**10**)  \
&emsp;&emsp;&emsp;&emsp;[x]~stl navigation~ (Untitled) https://www.notion.so/5c7f3dffdead43f0afb47770aace1e5f  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~generate complex task tree - for testing purposes~  \
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~show surrounding tasks~  \
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~select next task~  \
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp; * **Show all tasks [depth]** https://www.notion.so/fa60cc74a802420d87279ab1996a2604  
&emsp;&emsp; - Weekly review  \
&emsp;&emsp; - Particles  \


In [32]:
tree.show_selected(4)


 * **STL_17 sep 22** (Untitled) https://www.notion.so/483b8263ec2b48fa9bb941ba6e497516  
&emsp;&emsp;[x]~Duolingo~  
&emsp;&emsp; - T.assistant  
&emsp;&emsp; - Yoga Intensive  
&emsp;&emsp; - Jarvis  
&emsp;&emsp; * **Simple task lineariser** (Legend. Idea
Notebook: http://localhost:8888/notebooks/notion_assistant/dev/notebooks/Simple task linearisator-Copy1.ipynb
Key features
Find better usages
Improve functionality
Bugfixes
How to use this tool?
Untitled) https://www.notion.so/3299efe7e2264018a64e050c3e50f82c  
&emsp;&emsp;&emsp;&emsp;[x]~stl Notion~  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~Create template~  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~Load page from notion~  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~parse page from notion~ (**7**)  \
&emsp;&emsp;&emsp;&emsp;[x]~stl navigation~ (Untitled) https://www.notion.so/5c7f3dffdead43f0afb47770aace1e5f  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~generate complex task tree - for testing purposes~  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~show surrounding tasks~  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[x]~select next task~  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp; * **Show all tasks [depth]** https://www.notion.so/fa60cc74a802420d87279ab1996a2604  
&emsp;&emsp; - Weekly review  
&emsp;&emsp; - Particles  


In [120]:
# what I need to do now. Reverse order:
# Show selected task - * + Bold
# 






# Path: Better visualisation
# Extra:
# Show grey-out text for completed tasks
# Show line breaks properly
    # Option 1: Format repr somehow for md
    # Option 2: create a separate function for md.
    
    
    
    
# path: Navigation
# highlight selected task
# mark task as selected
# Select a task by name


In [49]:
# l = [1,2,3]
# l.index(2)

In [ ]:
# new features, ideas

# [done] Idea 1: display page links for selected task - near the name (or under?)

# Idea 2: show only the task tree for  

# [Done] Idea 3: Show task tree, starting from the root, including depth x.
    # and around selected.

# 
